# Week 4: Statistical Model Selection and Hypothesis Development
**Project:** Identifying Early Warning Signals of Diabetes Risk Using Routine Clinical Indicators

This notebook selects appropriate statistical models, performs feature selection, and develops testable hypotheses.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')

# Load and clean data (same preprocessing as Week 3)
df = pd.read_csv('diabetes.csv')
cols_with_invalid_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_invalid_zeros] = df[cols_with_invalid_zeros].replace(0, np.nan)
df.fillna(df.median(numeric_only=True), inplace=True)

print('Data loaded and cleaned. Shape:', df.shape)
df.head()

## 1. Understanding the Problem

Our target variable `Outcome` is **binary** (0 = No Diabetes, 1 = Diabetes).
This is a **supervised binary classification** problem.

### Model Selection Criteria
| Model | Suitable? | Reason |
|---|---|---|
| Linear Regression | ❌ No | Output is continuous — not appropriate for binary classification |
| ANOVA | ❌ No | Used for comparing means across groups, not for prediction |
| **Logistic Regression** | ✅ Yes | Designed for binary outcomes; interpretable coefficients; probabilistic output |
| **Random Forest** | ✅ Yes | Handles non-linear relationships; robust to outliers; provides feature importances |
| Gradient Boosting | ✅ Possible | Strong but more complex; better for later refinement |

We will **compare Logistic Regression and Random Forest** to select the best model.

## 2. Hypothesis Development

Based on EDA insights, we define the following statistical hypotheses:

**H₀ (Null Hypothesis):** There is no significant difference in Glucose, BMI, and Age between diabetic and non-diabetic patients.

**H₁ (Alternative Hypothesis):** Diabetic patients have significantly higher Glucose, BMI, and Age compared to non-diabetic patients.

**Significance level:** α = 0.05

**Test:** Two-sample independent t-test (to be conducted in Week 5)

## 3. Feature Selection

We use two complementary methods:
1. **Correlation with outcome** — simple linear relationship measure
2. **Logistic Regression coefficients** — importance in a linear model (requires scaling)
3. **Random Forest feature importances** — captures non-linear relationships

In [ ]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Method 1: Correlation with Outcome
corr_outcome = df.corr(numeric_only=True)['Outcome'].drop('Outcome').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['tomato' if v > 0 else 'steelblue' for v in corr_outcome]
corr_outcome.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.axhline(0.2, color='gray', linestyle='--', alpha=0.7, label='Threshold r=0.2')
ax.set_title('Pearson Correlation of Each Feature with Outcome', fontweight='bold')
ax.set_xlabel('Feature')
ax.set_ylabel('Correlation Coefficient')
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('Features with |r| > 0.2 (considered meaningful):')
print(corr_outcome[corr_outcome.abs() > 0.2].round(4).to_string())

In [ ]:
# Method 2: Logistic Regression Coefficients (scaled)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)

lr_coef = pd.Series(lr.coef_[0], index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['tomato' if v > 0 else 'steelblue' for v in lr_coef]
lr_coef.plot(kind='bar', ax=ax, color=colors, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression Coefficients (Scaled Features)', fontweight='bold')
ax.set_xlabel('Feature')
ax.set_ylabel('Coefficient')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('Logistic Regression Coefficients (higher = stronger positive association with diabetes):')
print(lr_coef.round(4).to_string())

In [ ]:
# Method 3: Random Forest Feature Importance
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
rf_importance.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Random Forest Feature Importances', fontweight='bold')
ax.set_xlabel('Feature')
ax.set_ylabel('Importance Score')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('Random Forest Feature Importances:')
print(rf_importance.round(4).to_string())

## 4. Feature Importance Comparison

In [ ]:
# Rank comparison across 3 methods
comparison = pd.DataFrame({
    'Correlation Rank': corr_outcome.abs().rank(ascending=False).astype(int),
    'LR Coefficient Rank': lr_coef.abs().rank(ascending=False).astype(int),
    'RF Importance Rank': rf_importance.rank(ascending=False).astype(int)
})
comparison['Average Rank'] = comparison.mean(axis=1).round(1)
comparison = comparison.sort_values('Average Rank')

print('Feature ranking across 3 selection methods (lower rank = more important):')
comparison

## 5. Baseline Model Comparison

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic Regression (needs scaling)
lr_scores = cross_val_score(LogisticRegression(max_iter=1000, random_state=42),
                             X_train_sc, y_train, cv=skf, scoring='roc_auc')

# Random Forest (no scaling needed)
rf_scores = cross_val_score(RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
                             X_train, y_train, cv=skf, scoring='roc_auc')

results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'CV AUC Mean': [lr_scores.mean().round(4), rf_scores.mean().round(4)],
    'CV AUC Std': [lr_scores.std().round(4), rf_scores.std().round(4)]
})

print('5-Fold Stratified Cross-Validation Results:')
print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(results['Model'], results['CV AUC Mean'],
       yerr=results['CV AUC Std'], color=['steelblue', 'tomato'],
       capsize=8, edgecolor='white')
ax.set_ylim(0.7, 0.9)
ax.set_title('Model Comparison: 5-Fold CV ROC-AUC', fontweight='bold')
ax.set_ylabel('ROC-AUC Score')
for i, (mean, std) in enumerate(zip(results['CV AUC Mean'], results['CV AUC Std'])):
    ax.text(i, mean + std + 0.005, f'{mean:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Selected Features and Model

Based on all three feature selection methods, the consistently top-ranked features are:

1. **Glucose** — strongest predictor across all methods
2. **BMI** — second most important
3. **Age** — consistently ranked third
4. **DiabetesPedigreeFunction** — genetic risk factor
5. **Pregnancies** — moderate importance

**Selected Model: Random Forest Classifier**
- Outperforms Logistic Regression in CV AUC
- Handles non-linear feature interactions
- Provides reliable feature importances
- Does not require feature scaling
- Produces probability outputs (`predict_proba`) for risk scoring

Logistic Regression will still be used in Week 5 for statistical validation and hypothesis testing, as its coefficients are more directly interpretable.

## Summary

| Decision | Choice | Justification |
|---|---|---|
| Problem type | Binary Classification | Outcome is 0 or 1 |
| Primary model | Random Forest | Higher AUC, handles non-linearity, feature importances |
| Analysis model | Logistic Regression | Interpretable for hypothesis testing |
| Top features | Glucose, BMI, Age, DPF, Pregnancies | Consistent across 3 selection methods |
| Evaluation metric | ROC-AUC (primary), F1-score (secondary) | Handles class imbalance better than accuracy |
| Data split strategy | Stratified 80/20 + 5-fold CV | Preserves class balance in each split |